# Adjacency Matrix Explorable

Build a directed graph from a hand-written adjacency matrix, compare row/column sums to degrees, symmetrize to undirected, then plant a weight, an isolate, and a self-loop.

In [1]:
import numpy as np
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# 5×5 directed adjacency matrix (rows = sources, columns = targets)
A = np.array([
    [0, 1, 0, 1, 0],  # 0 → 1, 3
    [0, 0, 1, 0, 0],  # 1 → 2
    [1, 0, 0, 1, 0],  # 2 → 0, 3
    [0, 0, 0, 0, 1],  # 3 → 4
    [0, 0, 0, 0, 0],  # 4 has no outgoing edges
])

print("Adjacency matrix A:")
print(A)

G_dir = nx.from_numpy_array(A, create_using=nx.DiGraph)

row_sums = A.sum(axis=1)
col_sums = A.sum(axis=0)
out_degrees = [G_dir.out_degree(n) for n in G_dir.nodes()]
in_degrees = [G_dir.in_degree(n) for n in G_dir.nodes()]

print("\nRow sums (out-degree by hand) vs NetworkX out_degree:")
for i in range(len(row_sums)):
    print(f"  Node {i}: row_sum={int(row_sums[i])}, out_degree={out_degrees[i]}")

print("\nColumn sums (in-degree by hand) vs NetworkX in_degree:")
for i in range(len(col_sums)):
    print(f"  Node {i}: col_sum={int(col_sums[i])}, in_degree={in_degrees[i]}")

Adjacency matrix A:
[[0 1 0 1 0]
 [0 0 1 0 0]
 [1 0 0 1 0]
 [0 0 0 0 1]
 [0 0 0 0 0]]

Row sums (out-degree by hand) vs NetworkX out_degree:
  Node 0: row_sum=2, out_degree=2
  Node 1: row_sum=1, out_degree=1
  Node 2: row_sum=2, out_degree=2
  Node 3: row_sum=1, out_degree=1
  Node 4: row_sum=0, out_degree=0

Column sums (in-degree by hand) vs NetworkX in_degree:
  Node 0: col_sum=1, in_degree=1
  Node 1: col_sum=1, in_degree=1
  Node 2: col_sum=1, in_degree=1
  Node 3: col_sum=2, in_degree=2
  Node 4: col_sum=1, in_degree=1


In [2]:
def draw_graph(G, title, pos=None):
    plt.figure(figsize=(7, 5))
    if pos is None:
        pos = nx.spring_layout(G, seed=7)
    is_directed = G.is_directed()
    nx.draw(
        G, pos,
        with_labels=True,
        node_color="lightyellow",
        node_size=600,
        font_size=10,
        arrows=is_directed,
        arrowstyle="-|>" if is_directed else "-",
        arrowsize=15,
        edge_color="gray",
    )
    edge_labels = nx.get_edge_attributes(G, "weight")
    if edge_labels:
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()
    return pos

pos = draw_graph(G_dir, "Directed graph from A")

/var/folders/8d/s4b0ksl144ndc6rnb11kz4380000gp/T/ipykernel_8964/468081677.py:22: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/var/folders/8d/s4b0ksl144ndc6rnb11kz4380000gp/T/ipykernel_8964/468081677.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Symmetrize: A + Aᵀ (clipped to 1) → undirected graph

Arrowheads should disappear when we draw the undirected version.

In [3]:
A_sym = np.clip(A + A.T, 0, 1).astype(int)
print("Symmetrized matrix (A + A.T clipped to 1):")
print(A_sym)

G_und = nx.from_numpy_array(A_sym)
draw_graph(G_und, "Undirected graph (symmetrized)", pos=pos)

Symmetrized matrix (A + A.T clipped to 1):
[[0 1 1 1 0]
 [1 0 1 0 0]
 [1 1 0 1 0]
 [1 0 1 0 1]
 [0 0 0 1 0]]


/Users/AlvaroPersonal/Library/Python/3.9/lib/python/site-packages/networkx/drawing/nx_pylab.py:305: UserWarning: 

The arrowstyle keyword argument is not applicable when drawing edges
with LineCollection.

To make this warning go away, either specify `arrows=True` to
force FancyArrowPatches or use the default value for arrowstyle.
Note that using FancyArrowPatches may be slow for large graphs.

  draw_networkx_edges(G, pos, arrows=arrows, **edge_kwds)
/var/folders/8d/s4b0ksl144ndc6rnb11kz4380000gp/T/ipykernel_8964/468081677.py:22: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/var/folders/8d/s4b0ksl144ndc6rnb11kz4380000gp/T/ipykernel_8964/468081677.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


{0: array([ 0.35330905, -0.10974329]),
 1: array([ 0.85488837, -0.72229287]),
 2: array([ 0.00957203, -0.40202222]),
 3: array([-0.46167012,  0.23405837]),
 4: array([-0.75609933,  1.        ])}

## Plant a weight, an isolate, and a self-loop

- **Weight**: non-zero entry > 1 (or fractional) on an edge — line thickness or label shows it.
- **Isolate**: a row and column of all zeros — node with no edges, drawn alone.
- **Self-loop**: non-zero diagonal entry — edge from a node back to itself.

In [4]:
# Extend to 6×6: add node 5 as isolate, weight on (0→2), self-loop on node 1
A_ext = np.zeros((6, 6), dtype=float)
A_ext[:5, :5] = A.astype(float)
A_ext[0, 2] = 3.0          # weighted edge 0 → 2
A_ext[1, 1] = 1.0          # self-loop at node 1
# node 5 stays all zeros → isolate

print("Extended matrix with weight, self-loop, and isolate:")
print(A_ext)

G_ext = nx.from_numpy_array(A_ext, create_using=nx.DiGraph)

print(f"\nIsolate node 5 — degree: {G_ext.degree(5)} (in={G_ext.in_degree(5)}, out={G_ext.out_degree(5)})")
print(f"Self-loop at node 1 — in={G_ext.in_degree(1)}, out={G_ext.out_degree(1)}")
print(f"Weight on edge (0,2): {G_ext[0][2].get('weight', 'missing')}")

pos_ext = nx.spring_layout(G_ext, seed=7)
draw_graph(G_ext, "Directed graph: weight (0→2), self-loop (1), isolate (5)")

Extended matrix with weight, self-loop, and isolate:
[[0. 1. 3. 1. 0. 0.]
 [0. 1. 1. 0. 0. 0.]
 [1. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]]

Isolate node 5 — degree: 0 (in=0, out=0)
Self-loop at node 1 — in=2, out=2
Weight on edge (0,2): 3.0


/var/folders/8d/s4b0ksl144ndc6rnb11kz4380000gp/T/ipykernel_8964/468081677.py:22: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/var/folders/8d/s4b0ksl144ndc6rnb11kz4380000gp/T/ipykernel_8964/468081677.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


{0: array([-0.11810527, -0.20420422]),
 1: array([ 0.0758725 , -0.41964306]),
 2: array([-0.17282122, -0.28780594]),
 3: array([-0.33677844,  0.00429169]),
 4: array([-0.44816756,  0.22605085]),
 5: array([1.        , 0.68131067])}